In [3]:
import pandas as pd
import numpy as np

listings = pd.read_csv("data/listings.csv")

listings.shape

(4574, 84)

## Part 2: Go deeper

### 1.

14.1% has no reviews (644). The highest share of no reviews is Hässelby-Vällingby (27%) and the lowest is Södermalms (7.6%)

In [4]:
listings["has_reviews"] = listings["number_of_reviews"] > 0

listings[["name", "has_reviews"]]

,name,has_reviews
0,Double room in central Stockholm with Wi-Fi,True
1,One room in appartement,True
2,Single room in central Stockholm with Wi-Fi,True
3,Penthouse in central Stockholm,True
4,Playful temple with genuine meetings for big&s...,True
...,...,...
4569,1 rum o kök Stockholm city,False
4570,Stylish Attic • Central Stockholm,False
4571,Cosy flat 10 mins from Stockholm city,False
4572,Room rental in a natural and central location.,False


In [5]:
listings["has_reviews"].value_counts(normalize=True).round(3)


has_reviews
True     0.859
False    0.141
Name: proportion, dtype: float64

In [6]:
listings["has_reviews"].value_counts()

has_reviews
True     3930
False     644
Name: count, dtype: int64

In [7]:

listings.groupby("neighbourhood_cleansed").agg(
    n_listings=("id", "count"),
    share_no_reviews=("has_reviews", lambda x:(~x).mean())
    ).round(3).sort_values("share_no_reviews", ascending=False)

,n_listings,share_no_reviews
neighbourhood_cleansed,,
Hässelby-Vällingby,148,0.270
Rinkeby-Tensta,108,0.259
Farsta,177,0.226
Skärholmens,96,0.208
Bromma,281,0.206
Enskede-Årsta-Vantörs,445,0.193
Spånga-Tensta,89,0.191
Norrmalms,469,0.154
Östermalms,297,0.141


### 2.

**The groups are not exacly the same size as the two reviews with the same value cannot be split**

In [8]:
reviewed = listings[listings["number_of_reviews"] >= 1]

reviewed.shape

(3930, 85)

In [9]:
reviewed["n_reviews_labeled"] = pd.qcut(reviewed["number_of_reviews"], 4, labels=['few', 'some', 'many', 'most'])

reviewed.shape

(3930, 86)

In [10]:
reviewed["n_reviews_labeled"].value_counts()

n_reviews_labeled
some    1032
few     1010
most     974
many     914
Name: count, dtype: int64

In [11]:
print(pd.qcut(reviewed["number_of_reviews"], 4).cat.categories)

IntervalIndex([(0.999, 4.0], (4.0, 13.0], (13.0, 36.0], (36.0, 1293.0]], dtype='interval[float64, right]')


### 3.

**We can conclude that there are a lot of high scores in general, but It is not clear that the score goes up with more reviews**

In [12]:
print(reviewed.groupby(["n_reviews_labeled"]).agg(n_reviews=("number_of_reviews", "count"), 
                                                  mean_score=("review_scores_rating", "mean"))
                                                  .round(2)
                                                  )


                   n_reviews  mean_score
n_reviews_labeled                       
few                     1010        4.79
some                    1032        4.84
many                     914        4.83
most                     974        4.80


### 4.

**The first list shows a couple of shared rooms on top which might not be what we are looking for when extracting such a list.**
**Also the very few number of listnings makes them dubious for any statisticts purposes, even if we wanted to count them as stated in the list.** 
**In short they are extreme values that we need to make a consious decision about before inclusion**

In [13]:
print(listings.groupby(["neighbourhood_cleansed", "room_type"]).agg(n_listings=("id", "count"), 
                                                                    avg_accommodates=("accommodates", "mean"))
                                                                    .round(2)
                                                                    .sort_values("avg_accommodates", ascending=False)
                                                                    )

                                        n_listings  avg_accommodates
neighbourhood_cleansed room_type                                    
Östermalms             Shared room               4              9.00
Kungsholmens           Shared room               8              6.50
Spånga-Tensta          Entire home/apt          54              5.33
Hässelby-Vällingby     Entire home/apt         118              5.25
Farsta                 Entire home/apt         145              4.97
Bromma                 Entire home/apt         259              4.90
Skarpnäcks             Entire home/apt         167              4.77
Älvsjö                 Entire home/apt         177              4.64
Enskede-Årsta-Vantörs  Entire home/apt         362              4.40
Skärholmens            Entire home/apt          56              4.29
Hägersten-Liljeholmens Entire home/apt         394              4.09
Kungsholmens           Entire home/apt         332              3.52
Rinkeby-Tensta         Entire home

In [14]:

neigh_roomtype_table = listings.groupby(["neighbourhood_cleansed", "room_type"]).agg(n_listings=("id", "count"), avg_accommodates=("accommodates", "mean")).round(2).sort_values("avg_accommodates", ascending=False)

print(neigh_roomtype_table[neigh_roomtype_table["n_listings"] >= 30])

                                        n_listings  avg_accommodates
neighbourhood_cleansed room_type                                    
Spånga-Tensta          Entire home/apt          54              5.33
Hässelby-Vällingby     Entire home/apt         118              5.25
Farsta                 Entire home/apt         145              4.97
Bromma                 Entire home/apt         259              4.90
Skarpnäcks             Entire home/apt         167              4.77
Älvsjö                 Entire home/apt         177              4.64
Enskede-Årsta-Vantörs  Entire home/apt         362              4.40
Skärholmens            Entire home/apt          56              4.29
Hägersten-Liljeholmens Entire home/apt         394              4.09
Kungsholmens           Entire home/apt         332              3.52
Rinkeby-Tensta         Entire home/apt          74              3.51
Södermalms             Entire home/apt        1004              3.44
Norrmalms              Entire home

### 5.

**Are there any types of listings that are more likley to be reviewed?**

In [15]:
print(listings.groupby(["room_type"]).agg(n_listings=("id", "count"), 
                                          avg_number_of_reviews=("number_of_reviews", "mean"))
                                          .round(2)
                                          .sort_values("avg_number_of_reviews", ascending=False)
                                          )

                 n_listings  avg_number_of_reviews
room_type                                         
Shared room              24                 114.21
Hotel room                3                  61.33
Private room            787                  40.54
Entire home/apt        3760                  34.23


In [16]:
mask = listings["room_type"] == "Shared room"

number_of_rev_per_type = listings[mask].groupby(["name", "room_type"]).agg(number_of_reviews=("number_of_reviews", "mean")).round(2).sort_values("number_of_reviews", ascending=False)

print(number_of_rev_per_type)

                                                               number_of_reviews
name                                              room_type                     
Shared  6 Bed Mixed Dorm Room                     Shared room              269.0
Generator - Bed in 4-bed Dorm                     Shared room              264.5
Loft in cosy apartment, Old Town                  Shared room              247.0
Bed in 8 Bed Shared Dorm Room                     Shared room              226.0
Generator - Bed in 6 Bed Dorm Female Only         Shared room              161.5
A bed in a Mixed 4 bed dorm room                  Shared room              147.0
A bed in a Mixed 12-bed dorm room                 Shared room              146.0
Generator - Bed in 6 Bed Dorm                     Shared room              146.0
Bed in 4 Bed Shared Capsule Room                  Shared room              118.0
A bed in an Mixed 8 bed dorm room                 Shared room              109.0
A bed in a Mixed 14-bed dorm

In [17]:
mask = listings["room_type"] == "Entire home/apt"

number_of_rev_per_type = listings[mask].groupby(["name", "room_type"]).agg(number_of_reviews=("number_of_reviews", "mean")).round(2).sort_values("number_of_reviews", ascending=False)

print(number_of_rev_per_type)

                                                                  number_of_reviews
name                                             room_type                         
Studio Apartment Double bed                      Entire home/apt             1293.0
Studio apartment in the heart of Södermalm       Entire home/apt              754.0
Top Location Apartment                           Entire home/apt              734.0
Studio Apartment two single beds                 Entire home/apt              683.0
Queen Studio Apartment with Sofa Bed             Entire home/apt              644.0
...                                                                             ...
Chic apartment 10-min walk from the city center  Entire home/apt                0.0
Whole apartment 1 bedroom                        Entire home/apt                0.0
Lovely apt in the ❤️ of sofo                     Entire home/apt                0.0
Ljus, mysig & välplanerad trea                   Entire home/apt            

#### Conclusions

**There are very few hotel rooms so our numbers are no use at all for them, this is unfortunatly (to a lesser extent) true for the Shared rooms which top the list.**
**I'd say that there are to few listings on the top 2 to draw any real conclusions, but with more listings it might be an interesting question.**
**Also, one needs to put some weight on the number of people that goes through the different accommodations and reformulate the question accordingly.**
**I didn't have time for that today, but maybe on friday!**

#### Friday addition:

**If we take "estimated_occupancy_l365d" into account we get a totally different result.**
**This could be for many reasons, but one particular piece that is missing is probably how long every stay is as there might well me a correlation between the number of nights that one stays and the type of accomodation. This is because it is likley that one only does a single review, nomatter how long the stay has been. So more work needed again :)**

In [ ]:
listings["rev_per_occupancy"] =  listings["estimated_occupancy_l365d"] / listings["number_of_reviews"]

print(listings.groupby(["room_type"]).agg(n_listings=("id", "count"), 
                                          avg_number_of_reviews_per_occ=("rev_per_occupancy", "mean"))
                                          .round(2)
                                          .sort_values("avg_number_of_reviews_per_occ", ascending=False)
                                          )

                 n_listings  avg_number_of_reviews_per_occ
room_type                                                 
Private room            787                           3.24
Entire home/apt        3760                           2.85
Shared room              24                           1.87
Hotel room                3                           0.47
